In [87]:
from aocd.models import Puzzle
import numpy as np
import pandas as pd
from itertools import combinations, pairwise
from collections import defaultdict

## Get Puzzle Input

In [2]:
year = 2025
day = 9
puzzle = Puzzle(year=year, day=day)

In [32]:
inp = puzzle.input_data

In [4]:
examples = puzzle.examples
examples

[Example(input_data='7,1\n11,1\n11,7\n9,7\n9,5\n2,5\n2,3\n7,3', answer_a='..............', answer_b='..............', extra=None)]

In [5]:
url = puzzle.url
url

'https://adventofcode.com/2025/day/9'

## Part A

In [31]:
# inp = examples[0].input_data

In [34]:
coords = pd.DataFrame(combinations(inp.split("\n"), 2)).rename(columns={0:"coords_1", 1:"coords_2"})
coords[["x1", "y1"]] = (coords["coords_1"].str.split(",", expand=True).astype(int))
coords[["x2", "y2"]] = (coords["coords_2"].str.split(",", expand=True).astype(int))
coords = coords.assign(
    area = lambda x: (np.abs(x.x2-x.x1)+1)*(np.abs(x.y1-x.y2)+1)
)

max_area = coords.area.max()

In [35]:
answer_a = max_area

## Part A Submission

In [36]:
puzzle.answer_a = answer_a

coerced int64 value np.int64(4771508457) to '4771508457'


## Part B

In [98]:
all_ends = inp.split("\n")
all_ends.append(all_ends[0])
ends = pd.DataFrame(all_ends).rename(columns={0:"coords"})
ends[["x", "y"]] = (ends["coords"].str.split(",", expand=True).astype(int))

xs = ends.sort_values(by="x").x.unique()
ys = ends.sort_values(by="y").y.unique()

coords_compressed_x = {x:i for i, x in enumerate(xs)}
coords_compressed_y = {x:i for i, x in enumerate(ys)}
ends["x"] = ends.x.map(coords_compressed_x)
ends["y"] = ends.y.map(coords_compressed_y)

edges = set()

## Rework this part to properly get all the edges. Something is off here.
rows = list(ends.itertuples(index=False))
for r1, r2 in zip(rows, rows[1:]):
    if r1.x == r2.x:
        for idx in range(min(r1.y,r2.y), max(r1.y,r2.y)+1):
            edges.add((r1.x, idx))
    elif r1.y == r2.y:
        for idx in range(min(r1.x,r2.x), max(r1.x,r2.x)+1):
            edges.add((idx, r1.y))
            
tmp = pd.DataFrame(edges).rename(columns={0:"x", 1:"y"})
gb_x = defaultdict(set)

for item in tmp.itertuples():
    gb_x[item.x].add(item.y)

for item in ends.itertuples():
    gb_x[item.x].add(item.y)

for x, ys in gb_x.items():
    ys = sorted(ys)
    for y1, y2 in zip(ys[::2], ys[1::2]):
        if y2 == y1+1:
            continue
        
        for idx in range(y1, y2+1):
            gb_x[x].add(idx)

In [99]:
def is_legal(x1,x2,y1,y2):
    x1 = coords_compressed_x[x1]
    x2 = coords_compressed_x[x2]
    y1 = coords_compressed_y[y1]
    y2 = coords_compressed_y[y2]
    for x in range(min(x1,x2), max(x1,x2)+1):
        if y1 not in gb_x[x]:
            return False
        if y2 not in gb_x[x]:
            return False
    for y in range(min(y1,y2), max(y1,y2)+1):
        if y not in gb_x[x1]:
            return False
        if y not in gb_x[x2]:
            return False
    return True

In [100]:
rows = list(coords.sort_values(by="area", ascending=False).itertuples(index=False))
for i in rows:
    if is_legal(i.x1,i.x2,i.y1,i.y2):
        area = i.area
        break

area

15851097

In [89]:
answer_b = area

## Part B Submission

In [90]:
puzzle.answer_b = answer_b

That's the right answer!  You are one gold star closer to decorating the North Pole.You have completed Day 9! You can [Shareon
  Bluesky
Twitter
Mastodon] this victory or [Return to Your Advent Calendar].
